# Chapter 4 — Discrete Behavior Cloning

Chapter 3 left us with a backbone that can *see* the brick and *read* the
instruction but cannot move the arm. This notebook builds the missing
piece — the **action head** — and trains it by behavior cloning on the
SO-101 pick-and-place demonstrations.

| Section | What you build | Figure |
| --- | --- | --- |
| 4.2 | Why regression collapses, and what a mixture fixes | 4.4 |
| 4.3 | A uniform per-control action tokenizer | — |
| 4.4 | Three action heads: factorized, autoregressive, parallel | — |
| 4.5 | One shared training loop for all three | — |
| 4.6 | What each head actually learned | 4.8, 4.9 |
| 4.7 | Bins back to motor commands, and how to schedule them | 4.10, 4.11 |

**Before you run:** *Runtime → Change runtime type → GPU*. The
autoregressive head decodes 96 positions in series and is very slow on
CPU.

In [ ]:
# Colab setup: install the three public chapter packages from GitHub.
import importlib.util
import subprocess
import sys

# Switch to 'main' after this feature branch is merged.
CHAPTER_4_REF = 'codex/ch04-colab-training-reporting'

if 'google.colab' in sys.modules:
    organization = 'Large-Robotics-Models-From-Scratch'
    chapter4_requirement = (
        f'lrm-ch04[data] @ git+https://github.com/{organization}/'
        f'lrm-code-chapter-4.git@{CHAPTER_4_REF}'
    )
    requirements = [
        f'lrm-ch02[data] @ git+https://github.com/{organization}/lrm-code-chapter-2.git@main',
        f'lrm-ch03 @ git+https://github.com/{organization}/lrm-code-chapter-3.git@main',
        chapter4_requirement,
    ]
    result = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '--quiet', *requirements],
        text=True, capture_output=True,
    )
    if result.returncode:
        detail = '\n'.join(
            part for part in (result.stdout, result.stderr) if part
        )
        raise RuntimeError(
            f'Chapter package installation failed:\n{detail}'
        )
    # lrm-ch04 stays at version 0.1.0 while the branch is under
    # development, so pip may keep an older 0.1.0 wheel in a reused
    # runtime. Refresh only this small package; keep resolved dependencies.
    refresh = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '--quiet',
         '--force-reinstall', '--no-deps', chapter4_requirement],
        text=True, capture_output=True,
    )
    if refresh.returncode:
        detail = '\n'.join(
            part for part in (refresh.stdout, refresh.stderr) if part
        )
        raise RuntimeError(
            f'Chapter 4 branch refresh failed:\n{detail}'
        )
    # LeRobot may upgrade Colab's preinstalled torch without upgrading its
    # optional torchaudio wheel. Transformers detects torchaudio by package
    # presence, then imports the incompatible binary on the way to SigLIP,
    # so `from ch03 import VLABackbone` dies with an undefined-symbol
    # OSError. Probe in a child process so a failed import cannot taint
    # this kernel; Chapters 2-4 use no audio, so remove only a broken wheel.
    audio_installed = importlib.util.find_spec('torchaudio') is not None
    audio_probe = subprocess.run(
        [sys.executable, '-c', 'import torch, torchaudio'],
        text=True, capture_output=True,
    )
    if audio_installed and audio_probe.returncode:
        uninstall = subprocess.run(
            [sys.executable, '-m', 'pip', 'uninstall', '--yes',
             'torchaudio'],
            text=True, capture_output=True,
        )
        if uninstall.returncode:
            detail = '\n'.join(
                part for part in (uninstall.stdout, uninstall.stderr)
                if part
            )
            raise RuntimeError(
                f'Could not remove incompatible torchaudio:\n{detail}'
            )
        print('Removed an ABI-incompatible optional torchaudio wheel.')
    api_probe = subprocess.run(
        [sys.executable, '-c',
         'from ch03 import VLABackbone; '
         'from ch04.cli import build_action_head; '
         'from ch04.decoding import decode_action_chunk, '
         'sample_action_grids'],
        text=True, capture_output=True,
    )
    if api_probe.returncode:
        detail = '\n'.join(
            part for part in (api_probe.stdout, api_probe.stderr) if part
        )
        raise RuntimeError(
            f'Chapter 4 API verification failed:\n{detail}'
        )
    stale = [
        name for name in sys.modules
        if name.split('.')[0] in ('ch02', 'ch03', 'ch04', 'transformers')
    ]
    for name in stale:
        sys.modules.pop(name, None)
    if stale:
        print('Cleared cached chapter modules. If the next cell still '
              'fails, use Runtime > Restart session and run again.')
    print('Installed Chapter 2, Chapter 3, and Chapter 4 packages.')

## Setup

Every figure uses thin lines, labelled axes, fixed colours per action
head, and distinct dash patterns that remain readable in grayscale.

In [ ]:
import itertools
import math
import os

import matplotlib.pyplot as plt
import numpy as np
import torch

from ch04.constants import ACTION_BINS, ACTION_DIM, ACTION_HORIZON
from ch04.style import use_manuscript_style

use_manuscript_style()

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
CHECKPOINT_ROOT = ('/content/ch04-checkpoints'
                   if os.path.isdir('/content') else './ch04-checkpoints')
TENSORBOARD_ROOT = f'{CHECKPOINT_ROOT}/tensorboard'

print(f'device            : {device}')
print(f'checkpoints       : {CHECKPOINT_ROOT}')
print(f'tensorboard logs  : {TENSORBOARD_ROOT}')
print(f'action grid       : H={ACTION_HORIZON} timesteps x '
      f'D={ACTION_DIM} controls')
print(f'bins per control  : {ACTION_BINS}')
print(f'chunk duration    : {ACTION_HORIZON / 30:.2f} s at 30 Hz')
print(f'loss at init      : ln({ACTION_BINS}) = '
      f'{math.log(ACTION_BINS):.3f} nats/token')

## 4.2 Why not just regress the action?

The instinctive design predicts six numbers and minimizes MSE. But
minimizing squared error is maximum likelihood under a Gaussian with fixed
variance, so the optimum is the **conditional mean** `E[a | s]`.

Below, the expert puts half its mass at −1 and half at +1, and the
observation gives no clue which. MSE averages them and lands in a valley
where the expert has essentially no data. A mixture keeps one component
per mode, with weights tracking how often each was demonstrated.

In [ ]:
from ch04.diagnostics import plot_bimodal_comparison
from ch04.exercises import (make_bimodal_actions, train_gmm_baseline,
                            train_mse_baseline)

observations, expert_actions = make_bimodal_actions()
mse_model, mse_history = train_mse_baseline()
mixture, gmm_history = train_gmm_baseline()

with torch.no_grad():
    collapsed = mse_model(torch.zeros(1, 1)).item()
    log_weights, means, sigmas = mixture(torch.zeros(1, 1))

support = float(((expert_actions - collapsed).abs() < 0.25).float().mean())
print(f'MSE prediction   : {collapsed:+.3f}')
print(f'  expert mass within 0.25 of it: {support:.1%}')
print(f'mixture means    : {[round(v, 3) for v in means[0].tolist()]}')
print(f'mixture weights  : '
      f'{[round(v, 3) for v in log_weights.exp()[0].tolist()]}')
print(f'mixture sigmas   : {[round(v, 3) for v in sigmas[0].tolist()]}')

# Figure 4.4
figure = plot_bimodal_comparison(
    expert_actions.numpy(), collapsed, mixture=mixture
)
figure.suptitle('Regression collapses between modes; a mixture does not')
plt.show()

## 4.3 Action tokenization

Instead of a mixture we take the categorical route: split each control's
range into `B = 256` bins and train with cross-entropy, exactly the
next-token machinery the language backbone already uses. Each marginal can
then hold several modes at once.

Set each control's bounds to its **1st and 99th percentiles** so outliers
do not stretch the bins and waste resolution. Fit these bounds on
**training episodes only**; the episode-disjoint validation split remains
unseen during quantization.

In [ ]:
from ch04 import ActionTokenizer
from ch04.data import (DEFAULT_DATASET_ID, collect_normalized_actions,
                       make_chunked_dataloaders)

train_loader, validation_loader, stats = make_chunked_dataloaders(
    DEFAULT_DATASET_ID, batch_size=4, validation_fraction=0.1)
normalized_actions = collect_normalized_actions(train_loader, stats)
tokenizer = ActionTokenizer.fit(normalized_actions)

example = normalized_actions[0]
bins = tokenizer.encode(example)
decoded = tokenizer.decode(bins)
half_bin = float(((tokenizer.hi - tokenizer.lo) / tokenizer.n_bins).max()) / 2

print(f'train episodes      : {len(train_loader.dataset.episodes)}')
print(f'validation episodes : {validation_loader.dataset.episodes}')
print()
print('one command through the full conversion path')
print(f'  normalized : {np.round(example, 4)}')
print(f'  bin ids    : {bins}')
print(f'  decoded    : {np.round(decoded, 4)}')
print(f'  error      : {np.abs(decoded - example).max():.5f} '
      f'(bound: {half_bin:.5f}, half a bin width)')

In [ ]:
# Visualize the quantization the policy will be trained against.
fig, axes = plt.subplots(1, 2, figsize=(11, 3.2))
axes[0].hist(normalized_actions[:, 0], bins=80, color='#4C72B0', alpha=0.75)
for bound, name in ((tokenizer.lo[0], 'q01'), (tokenizer.hi[0], 'q99')):
    axes[0].axvline(bound, color='#C44E52', ls='--', lw=1.4)
    axes[0].annotate(name, xy=(bound, 0.92), xycoords=('data', 'axes fraction'),
                     ha='center', fontsize=8, color='#C44E52')
axes[0].set(xlabel='normalized value, control 0', ylabel='training frames',
            title='Percentile bounds ignore the tails')

error = np.abs(tokenizer.decode(tokenizer.encode(normalized_actions))
               - normalized_actions)
axes[1].hist(error.ravel(), bins=60, color='#55A868', alpha=0.8)
axes[1].axvline(half_bin, color='#C44E52', ls='--', lw=1.4)
axes[1].annotate('half a bin', xy=(half_bin, 0.92),
                 xycoords=('data', 'axes fraction'), ha='right',
                 fontsize=8, color='#C44E52')
axes[1].set(xlabel='absolute round-trip error (normalized)',
            ylabel='control values',
            title='Quantization error is bounded by construction')
fig.tight_layout()
plt.show()

In [ ]:
# One target is an H x D grid: H future timesteps, D control dimensions.
demo_grid = torch.arange(ACTION_HORIZON * ACTION_DIM).reshape(
    1, ACTION_HORIZON, ACTION_DIM)
print(f'target grid                  : {tuple(demo_grid.shape)}')
print(f'parallel action positions    : {ACTION_HORIZON} '
      f'(one per timestep)')
print(f'autoregressive scalar tokens : {ACTION_HORIZON * ACTION_DIM} '
      f'(decoded in series)')
print(f'first two timestep vectors   : {demo_grid[0, :2].tolist()}')

## 4.4 Three ways to factor the grid

All three heads optimize the same `[H, D]` label grid and expose the same
`[H, D, B]` logits. They differ only in what each cell is allowed to
condition on:

| Head | Factorization | Cost |
| --- | --- | --- |
| **Factorized** | every cell from the state alone | one pass; cannot represent any dependency |
| **Parallel** | H bidirectional slots, one per timestep | one pass; still a product of per-cell marginals |
| **Autoregressive** | each cell on all earlier cells | exact conditioning; 96 serial steps |

Create a fresh Chapter 3 backbone for each head. A fixed seed and a fresh
loader generator pair the initialization and minibatch order. This remains
a single-seed comparison; a final paper-quality estimate should report
variation over multiple seeds.

In [ ]:
from ch03 import VLABackbone
from ch04.cli import build_action_head
from ch04.style import head_label

HEAD_NAMES = ('factorized', 'parallel', 'autoregressive')


def make_policy(name):
    """Return a fresh (backbone, head) pair for one head design."""
    backbone = VLABackbone().to(device)
    head = build_action_head(name, backbone).to(device)
    return backbone, head


for name in HEAD_NAMES:
    print(f'{name:>15} -> {head_label(name)}')

## 4.5 One training loop, three heads

The optimizer uses a larger learning rate for the newly initialized head
than for the pretrained trunk, SigLIP stays frozen, and the loss is
label-smoothed cross-entropy over the **non-padded** grid cells only.

`action_head_logits` absorbs the one difference between the heads — the
autoregressive branch needs the expert grid for teacher forcing — so a
single function trains and evaluates all three. The returned dictionary
keeps the scored batch for the diagnostics that follow.

In [ ]:
from ch04.analysis import set_seed
from ch04.data import action_targets, clone_dataloader, prepare_batch
from ch04.decoding import (decode_action_chunk, evaluate_open_loop,
                           evaluation_mode)
from ch04.diagnostics import plot_per_joint_metrics, plot_training_curves
from ch04.so101 import SO101_ACTION_NAMES
from ch04.train import (action_head_logits, held_out_metrics,
                        train_action_head)


def run_head_experiment(name, steps=None, eval_batches=None):
    # Defaults come from the configuration cell above, so the run size
    # can be changed in one place without touching this function.
    settings = globals()
    steps = settings['TRAIN_STEPS'] if steps is None else steps
    if eval_batches is None:
        eval_batches = settings['EVAL_BATCHES']
    mirror_root = settings.get('DRIVE_CHECKPOINT_ROOT')
    mirror_dir = (os.path.join(mirror_root, name)
                  if mirror_root is not None else None)
    resume_path = None
    if settings.get('RESUME_FROM_GOOGLE_DRIVE'):
        if mirror_dir is None:
            raise RuntimeError(
                'Google Drive resume requires a mounted Drive folder')
        resume_path = os.path.join(mirror_dir, 'latest.pt')
        if os.path.exists(resume_path):
            print(f'resuming {name} from {resume_path}')
        else:
            print(f'no Drive checkpoint for {name}; starting fresh')
            resume_path = None
    # Pair the initialization and shuffled minibatch order across heads.
    set_seed(settings['TRAIN_SEED'])
    experiment_loader = clone_dataloader(
        train_loader, shuffle=True, seed=settings['TRAIN_SEED'])
    backbone, head = make_policy(name)
    history = train_action_head(
        head, backbone, experiment_loader, stats, tokenizer, device,
        total_steps=steps,
        warmup_steps=min(settings['WARMUP_STEPS'], steps - 1),
        log_every=settings['LOG_EVERY'],
        checkpoint_every=settings['CHECKPOINT_EVERY'],
        validate_every=settings['VALIDATE_EVERY'],
        validation_rollout_batches=settings[
            'VALIDATION_ROLLOUT_BATCHES'],
        validation_loader=validation_loader,
        checkpoint_dir=f'{CHECKPOINT_ROOT}/{name}',
        checkpoint_mirror_dir=mirror_dir,
        resume_from=resume_path,
        tensorboard_log_dir=(f'{TENSORBOARD_ROOT}/{name}'
                             if settings['RUN_MODE'] == 'full' else None))

    batch = next(iter(validation_loader))
    model_inputs = prepare_batch(batch, stats, device, backbone)
    target_bins, _ = action_targets(batch, stats, tokenizer, device)
    with torch.no_grad(), evaluation_mode(backbone), evaluation_mode(head):
        logits = action_head_logits(
            head, backbone, model_inputs, target_bins)
    validation_metrics = held_out_metrics(
        head, backbone, validation_loader, stats, tokenizer, device,
        max_batches=eval_batches)
    validation_ce = validation_metrics['loss']

    prediction = decode_action_chunk(
        head, backbone, model_inputs, tokenizer, stats,
        strategy='argmax').cpu()
    metrics = evaluate_open_loop(
        head, itertools.islice(validation_loader, eval_batches),
        tokenizer, stats, backbone, device)

    plot_training_curves({name: history})
    plt.show()

    joint_figure = plot_per_joint_metrics(
        history, SO101_ACTION_NAMES, title=head_label(name))
    joint_figure.savefig(
        f'{CHECKPOINT_ROOT}/{name}/per_joint_metrics.png')
    if mirror_dir is not None:
        joint_figure.savefig(
            os.path.join(mirror_dir, 'per_joint_metrics.png'))
    plt.show()

    mae_std = metrics['mae_in_standard_deviations'].nanmean().item()
    print(f'{name}: held-out teacher-forced CE={validation_ce:.3f} '
          f'nats/token, teacher-forced accuracy='
          f'{validation_metrics["teacher_forced_accuracy"]:.1%}, '
          f'rollout accuracy={validation_metrics["accuracy"]:.1%} '
          f'(n={validation_metrics["rollout_token_count"]} tokens), '
          f'open-loop MAE={mae_std:.3f} training std')
    return {
        'head': head, 'backbone': backbone, 'history': history,
        'batch': batch, 'model_inputs': model_inputs,
        'target_bins': target_bins, 'validation_ce': validation_ce,
        'validation_metrics': validation_metrics,
        'open_loop_metrics': metrics,
        'mae_std': mae_std, 'prediction': prediction,
        'logits': logits.cpu(),
    }

### Choose a run mode — edit this cell, then run the three trainers

Everything that controls the three training runs lives here, so you can
set it once and leave the trainer cells alone. In Colab these render as
form fields; edit and re-run **this cell only**, then run 4.5.1 to 4.5.3.

**sanity** runs 10 optimizer steps per head and prints the loss after
every step. Use it to check that the pipeline executes; its metrics mostly
reflect initialization. **full** uses 20,000 steps per head and keeps
restartable checkpoints. It also writes TensorBoard scalars and a static
loss-curve PNG.

For a full run, leave **Save to Google Drive** enabled. Colab asks for
Drive permission once, then every 1,000-step local checkpoint is copied
atomically under `MyDrive/<folder>/<run name>/<head>/`. To continue after
a disconnect, keep the same folder and run name, enable **Resume from
Google Drive**, and re-run this cell followed by the trainer cells.

In [ ]:
RUN_MODE = 'sanity'        #@param ['sanity', 'full']
TRAIN_SEED = 17           #@param {type:"integer"}
EVAL_BATCHES = 32         #@param {type:"integer"}
VALIDATION_ROLLOUT_BATCHES = 8  #@param {type:"integer"}
PLOT_BATCHES = 64         #@param {type:"integer"}
OPEN_LOOP_BATCHES = 128   #@param {type:"integer"}
N_NEIGHBORS = 48          #@param {type:"integer"}
OPEN_LOOP_STEPS = 180     #@param {type:"integer"}
SAVE_TO_GOOGLE_DRIVE = True  #@param {type:"boolean"}
RESUME_FROM_GOOGLE_DRIVE = False  #@param {type:"boolean"}
DRIVE_FOLDER = 'lrm-book/ch04/checkpoints'  #@param {type:"string"}
DRIVE_RUN_NAME = 'chapter4-full-20k'  #@param {type:"string"}

MODES = {
    'sanity': dict(steps=10, warmup=5, log_every=1,
                   checkpoint_every=10, validate_every=2),
    'full': dict(steps=20_000, warmup=500, log_every=25,
                 checkpoint_every=1_000, validate_every=1_000),
}
assert RUN_MODE in MODES, "RUN_MODE must be 'sanity' or 'full'"
mode = MODES[RUN_MODE]
TRAIN_STEPS = mode['steps']
WARMUP_STEPS = mode['warmup']
LOG_EVERY = mode['log_every']
CHECKPOINT_EVERY = mode['checkpoint_every']
VALIDATE_EVERY = mode['validate_every']

if RESUME_FROM_GOOGLE_DRIVE and not SAVE_TO_GOOGLE_DRIVE:
    raise ValueError('Drive resume requires Drive checkpoint saving')
if RESUME_FROM_GOOGLE_DRIVE and RUN_MODE != 'full':
    raise ValueError('Drive resume is only available in full mode')
DRIVE_CHECKPOINT_ROOT = None
if RUN_MODE == 'full' and SAVE_TO_GOOGLE_DRIVE:
    if 'google.colab' not in sys.modules:
        print('Google Drive mirroring is available in Colab only.')
    else:
        from google.colab import drive
        drive.mount('/content/drive')
        drive_relative = os.path.join(
            DRIVE_FOLDER.strip('/'), DRIVE_RUN_NAME)
        DRIVE_CHECKPOINT_ROOT = os.path.join(
            '/content/drive/MyDrive', drive_relative)
        os.makedirs(DRIVE_CHECKPOINT_ROOT, exist_ok=True)

# Sanity logs all 10 updates. Full logs every 25 updates to both the
# notebook and TensorBoard without drawing a 20,000-point SVG.
# EVAL_BATCHES     held-out batches behind final rollout metrics.
# PLOT_BATCHES     held-out batches behind distribution diagnostics.
# OPEN_LOOP_BATCHES held-out batches scanned for a contiguous episode.
# N_NEIGHBORS      nearby held-out states in the softmax probe.
# OPEN_LOOP_STEPS  maximum frames in the representative episode window.

print(f'run mode          : {RUN_MODE}')
print(f'training seed     : {TRAIN_SEED} (single paired run)')
print(f'steps per head    : {TRAIN_STEPS:,}')
print(f'loss log interval : every {LOG_EVERY} step(s)')
print(f'total steps       : {3 * TRAIN_STEPS:,} across three heads')
print(f'final evaluation  : {EVAL_BATCHES} batches')
print(f'distribution scan : {PLOT_BATCHES} batches; '
      f'{N_NEIGHBORS} neighbours')
print(f'episode scan      : {OPEN_LOOP_BATCHES} batches; '
      f'up to {OPEN_LOOP_STEPS} displayed frames')
print(f'Drive backup      : {DRIVE_CHECKPOINT_ROOT or "disabled"}')
if device.type == 'cpu':
    print('\nRunning on CPU. Full mode requires a GPU runtime:\n'
          'Runtime > Change runtime type > GPU.')

### 4.5.1 Factorized head — the one-shot baseline

The factorized head uses one learned slot per grid cell. Its projected
state is broadcast into every slot and read out independently. The
independence
assumption is visible in what is *absent*: no slot's output enters another
slot's computation.

In [ ]:
results = {}
results['factorized'] = run_head_experiment('factorized')

### 4.5.2 Parallel head — bidirectional action slots

Append `H` learned action positions to the Chapter 3 prefix. The prefix
stays causal while the action block attends bidirectionally, and each of
the 16 slots predicts all six controls for one timestep in one pass. This
is a useful latency–quality baseline. Chapter 5 can improve its quality
with flow matching, but this discrete version still emits independent
per-cell marginals.

In [ ]:
results['parallel'] = run_head_experiment('parallel')

### 4.5.3 Autoregressive head — exact conditioning

Training is teacher-forced in one causal pass; inference feeds each
realized bin back through a KV cache. Its 96 serial decisions cost latency,
but exact conditioning solves the factorization problem most cleanly. It is
therefore the final discrete behavior-cloning policy used below.

In [ ]:
results['autoregressive'] = run_head_experiment('autoregressive')

# Use the final autoregressive discrete policy for deployment examples.
head = results['autoregressive']['head']
backbone = results['autoregressive']['backbone']
batch = results['autoregressive']['batch']
model_inputs = results['autoregressive']['model_inputs']

### Training curves, side by side

Three thin curves share one axis. Markers are held-out measurements; the
dotted line is `ln(256)`, the loss of a uniform policy. A curve pinned to
that line has not started learning; a training curve that falls while the
held-out markers do not is overfitting. Full mode also exposes the same
loss, accuracy, MAE, entropy, and learning-rate series through TensorBoard.

**Action-token accuracy** is exact argmax-bin accuracy over non-padded
action cells. Each joint has its own accuracy and MAE panel.

**MAE / training std** decodes each predicted bin to its centre, computes
`abs(predicted - demonstrated)` in raw action units, and divides by that
joint's training-set standard deviation. A value of 1.0 is an average
error of one training standard deviation. Train and held-out curve markers
are both teacher-forced and therefore comparable. Rollout accuracy and
open-loop MAE are reported separately from serial generation, with the
number of scored tokens printed alongside them. These metrics
measure held-out prediction, not closed-loop task success.

In [ ]:
training_figure = plot_training_curves(
    {name: results[name]['history'] for name in HEAD_NAMES})
training_curve_path = f'{CHECKPOINT_ROOT}/training_curves.png'
training_figure.savefig(training_curve_path)
if DRIVE_CHECKPOINT_ROOT is not None:
    training_figure.savefig(os.path.join(
        DRIVE_CHECKPOINT_ROOT, 'training_curves.png'))
plt.show()
print(f'saved static curves: {training_curve_path}')

if RUN_MODE == 'full':
    from tensorboard import notebook as tensorboard_notebook
    tensorboard_notebook.start(f'--logdir {TENSORBOARD_ROOT}')

## 4.6 What did it actually learn?

A falling loss says the policy puts more mass on expert bins. It does not
say whether a marginal kept both modes, or whether the cells go together.

### 4.6.1 Watching a softmax become bimodal

Search the held-out demonstrations for a locally ambiguous state: nearby
normalized proprioceptive states are scored for two
separated, balanced target-bin groups. The policy is never consulted during
selection. We then plot the demonstrated bins above the parallel policy's
marginals below. A bimodal **individual** softmax means genuine ambiguity
at that input; a bimodal **neighbourhood mean** alone may just mean the
nearby frames differ in image or object state. This is therefore a
proprioception-local dataset diagnostic, not proof that `p(a|full input)` is
multimodal. The cell scans three chunk offsets and all controls, then prints
the selected cell, anchor, modes, score, sample count, and seed.

In [ ]:
from ch04.analysis import (collect_action_softmaxes,
                           collect_expert_action_grids,
                           collect_expert_pairs,
                           collect_generated_pairs,
                           measure_inference_flops,
                           measure_inference_latency,
                           neighborhood_softmax_figure,
                           select_bimodal_anchor,
                           select_coupled_control_pair,
                           select_pair_mode_support, set_seed)
from ch04.diagnostics import (plot_joint_sample_panels,
                              plot_quality_compute_tradeoff)

SEED, TIMESTEP = 0, 0
set_seed(SEED)
if RUN_MODE != 'full':
    raise RuntimeError(
        'Publication figures require RUN_MODE=full or restored full checkpoints.')

def save_diagnostic_figure(figure, filename):
    path = os.path.join(CHECKPOINT_ROOT, filename)
    figure.savefig(path, bbox_inches='tight', dpi=180)
    if DRIVE_CHECKPOINT_ROOT is not None:
        figure.savefig(os.path.join(DRIVE_CHECKPOINT_ROOT, filename),
                       bbox_inches='tight', dpi=180)
    print(f'saved: {path}')

# Gather the bidirectional head once, then search cells without rerunning
# the backbone. Selection uses expert targets only.
probe_candidates = {}
parallel_result = results['parallel']
all_cells = collect_action_softmaxes(
    parallel_result['head'], parallel_result['backbone'],
    validation_loader, stats, tokenizer, device,
    max_batches=PLOT_BATCHES)
if all_cells['states'].shape[0] <= N_NEIGHBORS:
    raise RuntimeError('Increase PLOT_BATCHES: neighbour pool is too small.')
for timestep in (0, ACTION_HORIZON // 2, ACTION_HORIZON - 1):
    for control in range(ACTION_DIM):
        keep = all_cells['valid'][:, timestep, control]
        candidate = {
            'states': all_cells['states'][keep],
            'probabilities': all_cells['probabilities'][keep, timestep, control],
            'target_bins': all_cells['target_bins'][keep, timestep, control],
        }
        selection = select_bimodal_anchor(
            candidate['states'], candidate['target_bins'],
            n_neighbors=N_NEIGHBORS)
        probe_candidates[(timestep, control)] = (
            selection['score'], selection, candidate)

TIMESTEP, BASE_JOINT = max(
    probe_candidates, key=lambda key: probe_candidates[key][0])
_, softmax_selection, collected = probe_candidates[(TIMESTEP, BASE_JOINT)]
ANCHOR_INDEX = softmax_selection['anchor_index']
print('representative softmax probe:',
      dict(timestep=TIMESTEP, control=BASE_JOINT, anchor=ANCHOR_INDEX,
           peaks=softmax_selection['peak_bins'],
           separation=softmax_selection['separation_bins'],
           balance=softmax_selection['balance'],
           score=softmax_selection['score'],
           candidate_frames=collected['states'].shape[0], seed=SEED))

# Figure 4.8
softmax_figure = neighborhood_softmax_figure(
    collected, anchor_index=ANCHOR_INDEX,
    n_neighbors=N_NEIGHBORS,
    checkpoint=f'{CHECKPOINT_ROOT}/parallel/best.pt', seed=SEED)
save_diagnostic_figure(
    softmax_figure, 'figure_4_8_representative_softmax.png')
plt.show()

### 4.6.2 Measuring joint mismatch

Automatically choose the two held-out controls with the strongest
normalized mutual information. Selection uses demonstrations only, not
policy scores. Expert-derived gaps divide each control into two modes; the
four resulting combinations define demonstrated support. Every head then
contributes true deployment samples over the same held-out observations.
A coarse square-root-scaled histogram makes diffuse mass visible without
letting a few expert spikes wash out the policy panels.

The autoregressive samples condition on the model's own earlier choices, so
they are directly comparable with the one-pass heads. The title of each
policy panel reports its probability of an expert-unsupported combination.

In [ ]:
set_seed(SEED)
expert_grids = collect_expert_action_grids(
    validation_loader, stats, tokenizer, device,
    max_batches=PLOT_BATCHES)
pair_selection = select_coupled_control_pair(
    expert_grids['target_bins'], expert_grids['valid'],
    timestep=TIMESTEP, n_bins=ACTION_BINS)
PAIR_DIMS = pair_selection['dims']
print('representative coupled controls:', pair_selection)

expert_pairs = collect_expert_pairs(
    validation_loader, stats, tokenizer, device, dims=PAIR_DIMS,
    timestep=TIMESTEP, max_batches=PLOT_BATCHES)
mode_support = select_pair_mode_support(expert_pairs)
print('expert-derived mode support:', mode_support)
generated_pairs = {}
for name in HEAD_NAMES:
    result = results[name]
    generated_pairs[name] = collect_generated_pairs(
        result['head'], result['backbone'], validation_loader, stats,
        tokenizer, device, dims=PAIR_DIMS, timestep=TIMESTEP,
        max_batches=PLOT_BATCHES, strategy='sample')

# Figure 4.9
joint_figure = plot_joint_sample_panels(
    generated_pairs, expert_pairs, mode_support['splits'],
    mode_support['supported_quadrants'], n_bins=16,
    bin_range=(0, ACTION_BINS),
    dim_labels=(SO101_ACTION_NAMES[PAIR_DIMS[0]].replace('.pos', ''),
                SO101_ACTION_NAMES[PAIR_DIMS[1]].replace('.pos', '')))
save_diagnostic_figure(
    joint_figure, 'figure_4_9_representative_joint_mass.png')
plt.show()

for name in HEAD_NAMES:
    pairs = generated_pairs[name]
    quadrant = ((pairs[:, 0] >= mode_support['splits'][0]).astype(int) * 2
                + (pairs[:, 1] >= mode_support['splits'][1]).astype(int))
    rate = np.mean(~mode_support['supported_quadrants'][quadrant])
    print(f'{name:>15}: {rate:.1%} sampled mass off expert support '
          f'(n={len(pairs)})')

In [ ]:
# Measured forward-pass FLOPs for each head's real deployment path: one
# pass for the parallel heads, prefill plus every cached decode step for
# the autoregressive head. FLOPs alone understate AR, whose steps are
# dependent and cannot overlap, so the plot annotates serial depth too.
comparison_summary = {
    name: {
        'rollout_accuracy': results[name]['validation_metrics']['accuracy'],
    }
    for name in HEAD_NAMES
}
timing_batch = next(iter(validation_loader))
inference_flops, inference_latency = {}, {}
for name in HEAD_NAMES:
    result = results[name]
    inputs = prepare_batch(
        timing_batch, stats, device, result['backbone'])
    inference_flops.update(measure_inference_flops(
        {name: result['head']}, result['backbone'], inputs))
    inference_latency.update(measure_inference_latency(
        {name: result['head']}, result['backbone'], inputs,
        repeats=30, warmup=5))
print('| head | rollout acc | MAE/std | latency ms (p10–p90) | GFLOPs |')
print('| --- | ---: | ---: | ---: | ---: |')
for name in HEAD_NAMES:
    gflops = inference_flops[name]['flops'] / 1e9
    timing = inference_latency[name]
    accuracy = results[name]['validation_metrics']['accuracy']
    print(f"| {head_label(name)} | {accuracy:.1%} | "
          f"{results[name]['mae_std']:.3f} | "
          f"{timing['latency_ms']:.1f} "
          f"({timing['p10_ms']:.1f}–{timing['p90_ms']:.1f}) | "
          f"{gflops:.2f} |")

# Rollout token accuracy is the performance axis because it follows each
# head's deployment path. Latency is the cost axis: FLOPs score cached
# serial decoding BELOW one wide
# pass, so a FLOPs axis would put the AR head on the wrong side of the
# frontier. FLOPs are still measured above for the record.
tradeoff_figure = plot_quality_compute_tradeoff(
    comparison_summary, flops=inference_flops,
    latency=inference_latency)
save_diagnostic_figure(
    tradeoff_figure, 'section_4_6_quality_latency_tradeoff.png')
plt.show()

## 4.7 From distribution to motor commands

Argmax picks a learned **mode** rather than the mean of two modes. Per-cell
argmax can still assemble a combination the expert never demonstrated.
The tokenizer
returns normalized midpoints and Chapter 2's denormalizer converts those
to the dataset's raw command units. The plots below search held-out data
for one contiguous episode window with substantial expert motion. Selection
uses expert actions only, so it cannot favor a model that happens to look
good. Predictions are conditioned on recorded observations and never fed
back into the environment: this is open-loop error, not task success.

A second visualization starts from the recorded proprioceptive state,
treats one predicted chunk as absolute joint-position targets, computes
the resulting SO-101 end-effector path with the published URDF, and
projects that path onto the fixed side-camera view. Because the dataset
does not publish camera intrinsics or extrinsics, the projection uses a
documented weak-perspective fit and reports its anchor reprojection error.
It is an illustrative kinematic overlay, not a calibrated rollout.

In [ ]:
from ch04.analysis import (decoded_chunk_stream, open_loop_episode_trace,
                           select_representative_open_loop_window)
from ch04.camera_overlay import (SO101_URDF_URL, UrdfChain,
                                 end_effector_path, fit_affine_camera,
                                 plot_camera_trajectory_overlay)
from ch04.diagnostics import (plot_execution_schedules,
                              plot_open_loop_episode,
                              plot_open_loop_offset_errors)
from ch04.execution import execution_schedules

# The next cell builds Figures 4.10 and 4.11 from one canonical,
# sufficiently long held-out episode scan.

In [ ]:
# Figure 4.11: select one contiguous, high-motion held-out episode window
# from expert actions, then show the final AR policy against that reference.
open_loop_traces = {}
for name in HEAD_NAMES:
    result = results[name]
    trace = open_loop_episode_trace(
        result['head'], result['backbone'], validation_loader,
        tokenizer, stats, device, max_batches=OPEN_LOOP_BATCHES)
    open_loop_traces[name] = trace

if len(open_loop_traces['autoregressive']['expert']) < OPEN_LOOP_STEPS:
    raise RuntimeError('Increase OPEN_LOOP_BATCHES for the requested window.')
representative_window = select_representative_open_loop_window(
    open_loop_traces['autoregressive'], max_steps=OPEN_LOOP_STEPS,
    scale=stats['action']['std'].cpu().numpy())
print('representative open-loop window:',
      dict(episode=representative_window['episode_index'],
           start_frame=representative_window['start_frame'],
           end_frame=representative_window['end_frame'],
           motion_score=representative_window['motion_score']))

# Figure 4.10: three execution schedules over the AR chunk stream, with
# the recorded expert command as a reference rather than an implied target.
deployment = results['autoregressive']
chunks = decoded_chunk_stream(
    deployment['head'], deployment['backbone'], validation_loader,
    tokenizer, stats, device, max_batches=OPEN_LOOP_BATCHES)
schedule_figure = plot_execution_schedules(
    execution_schedules(chunks),
    expert=open_loop_traces['autoregressive']['expert'], control=0)
save_diagnostic_figure(
    schedule_figure, 'figure_4_10_execution_schedules.png')
plt.show()
episode_figure = plot_open_loop_episode(
    representative_window['predicted'],
    representative_window['expert'],
    representative_window['valid'],
    joint_names=SO101_ACTION_NAMES, head_name='autoregressive')
save_diagnostic_figure(
    episode_figure, 'figure_4_11_representative_open_loop_episode.png')
plt.show()

# Approximate camera-view trajectory from the same held-out start frame.
# The dataset actions are absolute joint-position targets in degrees.
# We prepend the measured proprioceptive state, apply one predicted chunk
# with a unit-gain position-servo assumption, and run URDF FK.
from pathlib import Path
from urllib.request import urlretrieve

urdf_path = Path(CHECKPOINT_ROOT) / 'so101_new_calib.urdf'
if not urdf_path.exists():
    urlretrieve(SO101_URDF_URL, urdf_path)
kinematic_chain = UrdfChain.from_file(urdf_path)

# Fixed side-camera fit from seven manually identified robot landmarks in
# episode 0, frame 0. These anchors make the approximation inspectable; the
# reported pixel RMSE measures only this affine fit, not camera calibration.
side_reference_state = np.array(
    [1.9561, -98.7437, 98.9243, 74.8198, -51.4530, 1.4094])
side_reference_links = [
    'base_link', 'shoulder_link', 'upper_arm_link', 'lower_arm_link',
    'wrist_link', 'gripper_link', 'gripper_frame_link']
side_reference_pixels = np.array([
    [300, 245], [335, 178], [378, 108], [240, 108],
    [400, 95], [420, 160], [452, 228]], dtype=float)
side_reference_world = np.stack([
    kinematic_chain.link_positions(side_reference_state)[link]
    for link in side_reference_links])
side_projection, side_anchor_rmse = fit_affine_camera(
    side_reference_world, side_reference_pixels)

overlay_index = int(representative_window['indices'][0])
overlay_sample = validation_loader.dataset[overlay_index]
side_image = torch.as_tensor(
    overlay_sample['observation.images.side']).cpu().numpy()
if side_image.shape[0] in (3, 4):
    side_image = np.moveaxis(side_image, 0, -1)
initial_state = torch.as_tensor(
    overlay_sample['observation.state']).cpu().numpy()
predicted_chunk = chunks[overlay_index].cpu().numpy()
world_path = end_effector_path(
    kinematic_chain, initial_state, predicted_chunk, tracking_alpha=1.0)
overlay_figure = plot_camera_trajectory_overlay(
    side_image, world_path, side_projection,
    calibration_rmse_px=side_anchor_rmse)
save_diagnostic_figure(
    overlay_figure, 'section_4_7_approximate_fk_camera_overlay.png')
plt.show()

# A shared-scale [prediction offset, control] error map retains the complete
# held-out comparison across all three heads.
offset_errors = {
    name: results[name]['open_loop_metrics'][
        'mae_in_standard_deviations'].cpu().numpy()
    for name in HEAD_NAMES
}
offset_figure = plot_open_loop_offset_errors(
    offset_errors, joint_names=SO101_ACTION_NAMES)
save_diagnostic_figure(
    offset_figure, 'section_4_7_open_loop_offset_errors.png')
plt.show()

for name in HEAD_NAMES:
    per_control_mae = torch.nanmean(
        results[name]['open_loop_metrics'][
            'mae_in_standard_deviations'], dim=0)
    summary = ', '.join(
        f'{joint}={value:.3f}' for joint, value in zip(
            SO101_ACTION_NAMES, per_control_mae.tolist()))
    tokens = results[name]['validation_metrics']['rollout_token_count']
    print(f'{head_label(name)} mean MAE/std by control (n={tokens} tokens): '
          f'{summary}')

### Export one chunk for a calibrated SO-101

This cell exports the autoregressive policy's first denormalized 16-step chunk,
including the training-data
range used for a safety check. Download the file and replay it from the
computer physically connected to the follower arm. The local command is
a dry run until `--execute` is supplied, and LeRobot caps each relative
joint target. Clear the workspace and keep an emergency stop ready.

The observation stays fixed while the chunk executes. Treat this motion
as a hardware smoke test, not a closed-loop task-success measurement.

In [ ]:
from ch04.so101 import export_action_chunk

SO101_CHUNK_PATH = f'{CHECKPOINT_ROOT}/autoregressive/so101_chunk.npz'
export_action_chunk(
    SO101_CHUNK_PATH, results['autoregressive']['prediction'][0].numpy(),
    fps=30, action_min=stats['action'].get('min'),
    action_max=stats['action'].get('max'),
    source=f'{CHECKPOINT_ROOT}/autoregressive/best.pt')
print(f'exported: {SO101_CHUNK_PATH}')
print('Local preview: ch04-so101-replay so101_chunk.npz '
      '--port <FOLLOWER_PORT>')
print('After inspection, repeat with --execute.')

## Run record

The quality–latency table and Pareto plot in Section 4.6 are the canonical
head comparison. This final printout repeats only its predictive metrics as
a quick check that the same evaluated objects reach the deployment cells.

In [ ]:
print(f"{'head':<16}{'TF CE':>10}{'rollout acc':>14}{'MAE / std':>12}")
for name in HEAD_NAMES:
    result = results[name]
    accuracy = result['validation_metrics']['accuracy']
    print(f"{name:<16}{result['validation_ce']:>10.3f}"
          f"{accuracy:>14.1%}{result['mae_std']:>12.3f}")

## Where to go next

- **Train for real.** Set `RUN_MODE = 'full'`, or from a terminal:
  `ch04-train --head all --steps 20000 --tensorboard-dir runs/ch04`.
  Figures regenerate from a
  checkpoint with
  `ch04-figures checkpoints/parallel/best.pt --head parallel`.
- **Sweep the bin count.** Coarser bins raise quantization error; finer
  bins leave fewer examples per class. 256 is a default to validate per
  task, not a constant.
- **Try the decoding rules.** `decode_action_chunk(..., strategy='sample',
  temperature=..., top_p=...)` — temperature sets how peaked the
  distribution is, top-p decides how much of the tail survives.
- **Interpret the diagnostics correctly.** Open-loop agreement measures
  prediction error on held-out demonstrations, not closed-loop task
  success. Better sampling does not remove joint mismatch from independent
  categorical outputs; a joint generative action model is needed for that.